# Word Embedding and Keras Embedding Layer

This notebook demonstrates the basic text-to-vector pipeline used before an RNN processes text.

### What we will do

1. Prepare a few sample sentences
2. Define a vocabulary size
3. Convert words into integer token IDs
4. Apply sequence padding
5. Build a Keras `Embedding` layer
6. Inspect the embedding output and tensor shapes

> **Note:** The `one_hot()` utility used in this notebook produces integer token IDs for the words. It is not creating an explicit 10,000-dimensional one-hot vector for every word.

## 1. Import Required Libraries

In [56]:
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import Embedding
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras import Sequential, Input
import numpy as np

## 2. Create Sample Sentences

We start with a small set of sentences so that the complete embedding pipeline is easy to understand.

The sentences intentionally have different lengths; this will help us understand why sequence padding is required.

In [58]:
# sentences
sentences = ['the glass of milk',
             'the glass of juice',
             'the cup of tea',
             'I am a good boy',
             'I am a good developer',
             'understand the meaning of words',
             'your videos are good',]

sentences

['the glass of milk',
 'the glass of juice',
 'the cup of tea',
 'I am a good boy',
 'I am a good developer',
 'understand the meaning of words',
 'your videos are good']

## 3. Define the Vocabulary Size

`vocab_size` represents the number of token IDs available for this demonstration.

We use a vocabulary size of **10,000**.

In [59]:
vocab_size = 10_000
vocab_size

10000

## 4. Convert Words into Integer Token IDs

Each sentence is converted into a sequence of integer IDs.

These IDs are the inputs that will be passed to the Embedding layer in the next stage.

In [60]:
encoded_sequences = [
    one_hot(sentence, vocab_size)
    for sentence in sentences
]

encoded_sequences

[[9118, 3805, 3731, 7702],
 [9118, 3805, 3731, 833],
 [9118, 4090, 3731, 7632],
 [9190, 8659, 7958, 1120, 7877],
 [9190, 8659, 7958, 1120, 4472],
 [3517, 9118, 3935, 3731, 5443],
 [616, 9675, 6305, 1120]]

### Inspect One Encoded Sentence

The output is a sequence of integers rather than the original words.

For example, a sentence such as `"the glass of milk"` becomes a sequence similar to:

`[6186, 6775, 637, 4895]`

The exact values depend on the tokenizer/hash mapping used by the utility.

In [61]:
encoded_sequences[0]

[9118, 3805, 3731, 7702]

In [67]:
# # One Hot Representation
# one_hot_repr = [one_hot(words, vocab_size)for words in sentences]
# one_hot_repr

## 5. Apply Sequence Padding

Different sentences contain different numbers of tokens. Neural network batches require a consistent sequence length.

We therefore use:

- `maxlen = 8`
- `padding = "pre"`

With **pre-padding**, zeros are added to the beginning of shorter sequences.

In [63]:
max_sequence_length = 8

padded_sequences = pad_sequences(
    encoded_sequences,
    maxlen=max_sequence_length,
    padding="pre"
)

print("Padded shape:", padded_sequences.shape)
padded_sequences

Padded shape: (7, 8)


array([[   0,    0,    0,    0, 9118, 3805, 3731, 7702],
       [   0,    0,    0,    0, 9118, 3805, 3731,  833],
       [   0,    0,    0,    0, 9118, 4090, 3731, 7632],
       [   0,    0,    0, 9190, 8659, 7958, 1120, 7877],
       [   0,    0,    0, 9190, 8659, 7958, 1120, 4472],
       [   0,    0,    0, 3517, 9118, 3935, 3731, 5443],
       [   0,    0,    0,    0,  616, 9675, 6305, 1120]], dtype=int32)

### Inspect the First Padded Sequence

The original first sentence contains four tokens. Because the target length is eight, four zeros are added at the beginning.

In [64]:
padded_sequences[0]

array([   0,    0,    0,    0, 9118, 3805, 3731, 7702], dtype=int32)

## 6. Define the Embedding Dimension

The embedding dimension determines how many numerical features represent each token.

For this demonstration we use an embedding dimension of **10**.

So, each token ID will be mapped to a vector of length 10.

In [65]:
embedding_dim = 10

In [66]:
# sent_length = 8
# embedded_docs = pad_sequences(one_hot_repr, padding='pre', maxlen=sent_length)
# print(embedded_docs)

## 7. Build the Embedding Layer

The Keras `Embedding` layer maps each integer token ID to a dense vector.

For this model:

- Vocabulary size = `10,000`
- Sequence length = `8`
- Embedding dimension = `10`

Therefore, a batch of padded sequences with shape:

`(batch_size, 8)`

is transformed into:

`(batch_size, 8, 10)`

In [68]:
model = Sequential([
    Input(shape=(max_sequence_length,), dtype="int32"),
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim
    )
])

model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 8, 10)          │       100,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 100,000 (390.62 KB)

 Trainable params: 100,000 (390.62 KB)

 Non-trainable params: 0 (0.00 B)

In [47]:
# model = Sequential()
# model.add(Embedding(voc_size, dim, input_length=sent_length))
# model.compile('adam', 'mse')

In [48]:
model = Sequential([
    Input(shape=(sent_length,), dtype="int32"),
    Embedding(
        input_dim=voc_size,
        output_dim=dim
    )
])

model.compile(
    optimizer="adam",
    loss="mse"
)

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 8, 10)          │       100,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 100,000 (390.62 KB)

 Trainable params: 100,000 (390.62 KB)

 Non-trainable params: 0 (0.00 B)

### Understanding the Parameter Count

The Embedding layer contains one trainable vector for each vocabulary entry.

Therefore:

`10,000 × 10 = 100,000` trainable parameters.

In [69]:
expected_embedding_params = vocab_size * embedding_dim
print("Expected embedding parameters:", expected_embedding_params)

Expected embedding parameters: 100000


## 8. Generate Embedding Representations

Now we pass the padded integer sequences through the Embedding layer.

The output contains a 10-dimensional vector for every token position.

In [70]:
embedding_output = model.predict(padded_sequences, verbose=0)

print("Input shape :", padded_sequences.shape)
print("Output shape:", embedding_output.shape)

Input shape : (7, 8)
Output shape: (7, 8, 10)


### Inspect One Sample

For one sample, the input shape is `(1, 8)` and the embedding output shape is `(1, 8, 10)`.

The first dimension is the batch size, the second is the sequence length, and the third is the embedding dimension.

In [71]:
sample_input = padded_sequences[0:1]
sample_embedding = model.predict(sample_input, verbose=0)

print("Single-sample input shape :", sample_input.shape)
print("Single-sample output shape:", sample_embedding.shape)

sample_embedding[0]

Single-sample input shape : (1, 8)
Single-sample output shape: (1, 8, 10)


array([[-0.03319307,  0.03025192,  0.02751019, -0.04328047, -0.04416726,
         0.03674087, -0.01191346, -0.00748247, -0.00562247,  0.04391602],
       [-0.03319307,  0.03025192,  0.02751019, -0.04328047, -0.04416726,
         0.03674087, -0.01191346, -0.00748247, -0.00562247,  0.04391602],
       [-0.03319307,  0.03025192,  0.02751019, -0.04328047, -0.04416726,
         0.03674087, -0.01191346, -0.00748247, -0.00562247,  0.04391602],
       [-0.03319307,  0.03025192,  0.02751019, -0.04328047, -0.04416726,
         0.03674087, -0.01191346, -0.00748247, -0.00562247,  0.04391602],
       [ 0.0466175 ,  0.00243204,  0.03649108,  0.00559404, -0.03269871,
        -0.00052046, -0.03053721, -0.02234179, -0.00375569, -0.00592879],
       [-0.03805474, -0.03587054, -0.04789584, -0.04512617,  0.01581854,
        -0.03170489,  0.03814967, -0.02647441,  0.00328983, -0.02644001],
       [-0.00773847, -0.01059737,  0.00482948, -0.04668843,  0.04441616,
        -0.03545173, -0.04733358,  0.04249949

## 9. Key Takeaways

- Text must be converted into numerical token IDs before entering the neural network.
- Padding makes variable-length sequences compatible with a fixed input length.
- The Embedding layer converts token IDs into dense vectors.
- With `embedding_dim = 10`, every token is represented by 10 numerical features.
- For `vocab_size = 10,000` and `embedding_dim = 10`, the Embedding layer contains `100,000` trainable parameters.
- The Embedding layer becomes useful for the downstream RNN because the RNN receives dense vector representations instead of raw text.